In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dhg
from dhg.nn import GCNConv

# 1. 构建超图 (5个原始节点, 2条超边)
hg = dhg.Hypergraph(5, [(0, 1, 2), (2, 3, 4)]) # type: ignore

# 2. 星形扩展：得到普通图 g 和 顶点掩码 v_mask
g, v_mask = dhg.Graph.from_hypergraph_star(hg) # type: ignore
# 此时 g 有 7 个节点 (0-4 为真实, 5-6 为虚拟)

# 3. 构造初始特征 (假设原始特征维度为 4)
# 真实节点用随机特征，虚拟节点通常初始化为 0 或全局平均值
X = torch.zeros(g.num_v, 4)
X[v_mask] = torch.randn(5, 4)

/home/zhoutong/projects/dhg/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class StarGCN(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hid_ch)
        self.conv2 = GCNConv(hid_ch, out_ch)

    def forward(self, x, g):
        x = F.relu(self.conv1(x, g))
        x = self.conv2(x, g)
        return x

# 实例化模型 (预测 2 个类别)
model = StarGCN(4, 8, 2)

In [3]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# 假设真实节点的标签
target = torch.tensor([0, 1, 0, 1, 0]) 

model.train()
for epoch in range(10):
    optimizer.zero_grad()
    
    # 预测所有节点 (7个)
    out = model(X, g)
    
    # 【关键】只取出真实节点的预测值进行计算
    real_node_out = out[v_mask] 
    
    loss = F.cross_entropy(real_node_out, target)
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch}: Loss {loss.item():.4f}")

Epoch 0: Loss 0.7578
Epoch 1: Loss 0.6964
Epoch 2: Loss 0.6677
Epoch 3: Loss 0.7203
Epoch 4: Loss 0.6931
Epoch 5: Loss 0.7067
Epoch 6: Loss 0.6894
Epoch 7: Loss 0.7082
Epoch 8: Loss 0.6619
Epoch 9: Loss 0.6931


## 1. 核心原理：生活中的“中转站”比喻
想象一个超图：“张三、李四、王五”三个人共同参加了一个“AI研讨会”。

在超图中，这三个人（节点）被一个圈（超边）围在一起。

但在普通图中，一个人只能连另一个人。

**星形扩展(Star Expansion)**的解决办法是：
引入一个“虚拟节点”——研讨会本身。

让张三、李四、王五分别连接到“研讨会”。

张三想给王五传达信息时，先传给“研讨会”，再由“研讨会”传给王五。

+ 第一步：降维打击（构建扩展图）

    - 原理：超图不能直接跑 GCN。这一步把超图变成了普通图。

    — 结果：如果原图有 5 个节点（0-4），2 条超边。扩展后总节点变成 7 个。

    - 节点 0-4：真实人类。

    - 节点 5-6：虚拟的“研讨会”。

    - v_mask：这是一个布尔列表 [True, True, True, True, True, False, False]。它像一张通行证，告诉模型谁是真人，谁是虚拟人。
+ 第二步：特征初始化
    - 原理：模型需要输入每个人的特征（比如他们的专业知识）。

    - 细节：我们给真人（v_mask 为 True 的地方）分配了随机数。虚拟节点（研讨会）暂时设为 0。


+ 第三步：消息传递（GCN 卷积）
  + 原理：这是最神奇的地方。
    + 第一层卷积：真人节点把信息传给虚拟节点。此时，虚拟节点吸收了所有成员的信息。
    + 第二层卷积：虚拟节点把吸收到的汇总信息再传回给真人节点。
    +  结果：经过两层卷积，原本不相连的张三和王五完成了信息交换！



In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dhg
from dhg.data import Cora
from dhg.nn import GCNConv

# 1. 加载数据
data = Cora()
X, y = data['features'], data['labels']
train_mask, test_mask = data['train_mask'], data['test_mask']
num_v, num_classes = data['num_vertices'], data['num_classes']

# 2. 构建超图并进行星形扩展
base_g = dhg.Graph(num_v, data['edge_list'])
hg = dhg.Hypergraph.from_graph_kHop(base_g, k=1)
g_star, v_mask = dhg.Graph.from_hypergraph_star(hg)

# 3. 定义 Star-GCN 模型
class StarGCN(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hid_ch)
        self.conv2 = GCNConv(hid_ch, out_ch)

    def forward(self, x, g, v_mask):
        # 构造扩展特征矩阵：虚拟节点特征设为 0
        full_X = torch.zeros(g.num_v, x.shape[1]).to(x.device)
        full_X[v_mask] = x
        
        # 消息传递
        x_out = F.relu(self.conv1(full_X, g))
        x_out = self.conv2(x_out, g)
        return x_out # 返回包含虚拟节点的输出

# 4. 初始化与训练
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StarGCN(X.shape[1], 16, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.02, weight_decay=5e-4)

# 5. 训练循环
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    out = model(X.to(device), g_star.to(device), v_mask)
    
    # 仅计算真实节点的 loss
    loss = F.cross_entropy(out[v_mask][train_mask], y[train_mask].to(device))
    loss.backward()
    optimizer.step()
    
    # 测试准确率
    model.eval()
    pred = out.argmax(dim=1)
    acc = (pred[v_mask][test_mask] == y[test_mask].to(device)).float().mean()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} | Acc: {acc:.2%}")

Epoch 00 | Loss: 1.9553 | Acc: 11.00%
Epoch 10 | Loss: 1.9324 | Acc: 17.10%
Epoch 20 | Loss: 1.8956 | Acc: 23.80%
Epoch 30 | Loss: 1.8501 | Acc: 25.90%
Epoch 40 | Loss: 1.8026 | Acc: 25.10%
Epoch 50 | Loss: 1.7002 | Acc: 26.80%
Epoch 60 | Loss: 1.7628 | Acc: 29.80%
Epoch 70 | Loss: 1.7163 | Acc: 30.20%
Epoch 80 | Loss: 1.6572 | Acc: 32.50%
Epoch 90 | Loss: 1.6735 | Acc: 33.60%
Epoch 100 | Loss: 1.6000 | Acc: 31.70%
Epoch 110 | Loss: 1.5277 | Acc: 32.60%
Epoch 120 | Loss: 1.5901 | Acc: 32.20%
Epoch 130 | Loss: 1.5080 | Acc: 33.40%
Epoch 140 | Loss: 1.6295 | Acc: 31.80%
Epoch 150 | Loss: 1.5453 | Acc: 31.60%
Epoch 160 | Loss: 1.5915 | Acc: 32.70%
Epoch 170 | Loss: 1.5117 | Acc: 32.80%
Epoch 180 | Loss: 1.4946 | Acc: 33.10%
Epoch 190 | Loss: 1.5114 | Acc: 32.40%
